# 📊 EcoSense — Data Generation
## Synthetic Environmental Sensor Data

This notebook generates realistic time-series data simulating environmental monitoring equipment.

**Sensors simulated:**
- **Water Quality:** pH, Turbidity, Dissolved Oxygen, Temperature, Conductivity
- **Air Quality:** CO₂, NOₓ, SO₂, PM2.5

**Features:**
- Daily cycles & seasonal patterns
- Factory shift variations
- Inter-sensor correlations
- Realistic anomaly injection (drift, spikes, flatlines, cyclic irregularities)
- Missing data simulation

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generator import generate_dataset, SENSOR_CONFIG

# Plot styling
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('EcoSense Data Generator ready!')

## 1. Generate Dataset

In [ ]:
# Generate 6 months of data at 15-minute intervals
data = generate_dataset(
    start_date='2026-01-01',
    end_date='2026-06-30',
    interval_minutes=15,
    master_seed=42
)

In [ ]:
# Extract DataFrames
water_df = data['water_quality']
air_df = data['air_quality']
anomaly_log = data['anomaly_log']

print(f'Water Quality shape: {water_df.shape}')
print(f'Air Quality shape: {air_df.shape}')
print(f'Anomalies injected: {len(anomaly_log)}')

## 2. Quick Preview

In [ ]:
print('=== Water Quality Data ===')
water_df.head(10)

In [ ]:
print('=== Air Quality Data ===')
air_df.head(10)

In [ ]:
print('=== Anomaly Log ===')
anomaly_log

## 3. Quick Visualization

In [ ]:
water_sensors = ['pH', 'turbidity_ntu', 'dissolved_oxygen_mg_l', 'temperature_c', 'conductivity_us_cm']

fig, axes = plt.subplots(len(water_sensors), 1, figsize=(16, 3 * len(water_sensors)), sharex=True)
fig.suptitle('Water Quality Sensors — 6 Month Overview', fontsize=16, fontweight='bold', y=1.01)

for idx, sensor in enumerate(water_sensors):
    ax = axes[idx]
    ts = pd.to_datetime(water_df['timestamp'])
    ax.plot(ts, water_df[sensor], linewidth=0.5, alpha=0.8, color='steelblue')
    
    # Highlight anomalies
    anomaly_col = f'is_anomaly_{sensor}'
    if anomaly_col in water_df.columns:
        anomaly_mask = water_df[anomaly_col] == 1
        ax.scatter(ts[anomaly_mask], water_df[sensor][anomaly_mask], 
                  color='red', s=2, alpha=0.5, label='Anomaly')
    
    config = SENSOR_CONFIG['water_quality'][sensor]
    ax.axhline(y=config['normal_min'], color='green', linestyle='--', alpha=0.4, linewidth=0.8)
    ax.axhline(y=config['normal_max'], color='green', linestyle='--', alpha=0.4, linewidth=0.8)
    ax.set_ylabel(f"{sensor}\n({config['unit']})", fontsize=9)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.savefig('../reports/water_quality_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
air_sensors = ['co2_ppm', 'nox_ppm', 'so2_ppm', 'pm25_ug_m3']

fig, axes = plt.subplots(len(air_sensors), 1, figsize=(16, 3 * len(air_sensors)), sharex=True)
fig.suptitle('Air Quality Sensors — 6 Month Overview', fontsize=16, fontweight='bold', y=1.01)

for idx, sensor in enumerate(air_sensors):
    ax = axes[idx]
    ts = pd.to_datetime(air_df['timestamp'])
    ax.plot(ts, air_df[sensor], linewidth=0.5, alpha=0.8, color='darkorange')
    
    anomaly_col = f'is_anomaly_{sensor}'
    if anomaly_col in air_df.columns:
        anomaly_mask = air_df[anomaly_col] == 1
        ax.scatter(ts[anomaly_mask], air_df[sensor][anomaly_mask],
                  color='red', s=2, alpha=0.5, label='Anomaly')
    
    config = SENSOR_CONFIG['air_quality'][sensor]
    ax.axhline(y=config['normal_min'], color='green', linestyle='--', alpha=0.4, linewidth=0.8)
    ax.axhline(y=config['normal_max'], color='green', linestyle='--', alpha=0.4, linewidth=0.8)
    ax.set_ylabel(f"{sensor}\n({config['unit']})", fontsize=9)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.savefig('../reports/air_quality_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Data generation and initial visualization complete!')
print('Proceed to notebook 02_eda.ipynb for detailed analysis.')